# Этап 2.2. Источники данных и аналитический портрет
цена и спрос на рынке электроэнергии Германии (зона DE-LU). Для изучения берём данные за 2024 год.

| № | Источник | Тип по п.3 | Что даёт |
|:-:|:--|:--|:--|
| 1 | Open Power System Data | реляционный (SQLite) | цена, нагрузка, ветер, солнце |
| 2 | ENTSO-E Transparency Platform | XML | цена, нагрузка, генерация (нужен токен, пока не получен) |
| 2a | Eurostat | XML | производство электроэнергии по видам топлива и странам |
| 3 | Energy-Charts | API, JSON | цена, нагрузка, генерация по типам |
| 4 | Open-Meteo | API, JSON | погода |
| 5 | Nager.Date | API, JSON | праздники |
| 6 | Google News | текст | новости о ценах на электроэнергию |

In [1]:
import os, sqlite3, urllib.request
import xml.etree.ElementTree as ET
from io import StringIO
import pandas as pd
import requests

## 1. Open Power System Data — реляционный набор (SQLite)

Почасовые данные по энергосистемам Европы за 2015–2020 гг., файл базы SQLite (226 МБ). Названия столбцов устроены как `регион_переменная_атрибут`: например, `DE_LU_price_day_ahead` — зона DE-LU, цена, рынок на сутки вперёд.

In [2]:
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/opsd.sqlite'):
    urllib.request.urlretrieve('https://data.open-power-system-data.org/time_series/2020-10-06/time_series.sqlite', 'data/opsd.sqlite')

con = sqlite3.connect('data/opsd.sqlite')
opsd = pd.read_sql('''
    SELECT utc_timestamp, DE_LU_price_day_ahead, DE_load_actual_entsoe_transparency,
           DE_solar_generation_actual, DE_wind_generation_actual
    FROM time_series_60min_singleindex
    WHERE DE_LU_price_day_ahead IS NOT NULL
''', con)
opsd.head()

,utc_timestamp,DE_LU_price_day_ahead,DE_load_actual_entsoe_transparency,DE_solar_generation_actual,DE_wind_generation_actual
0,2018-09-30T23:00:00Z,56.10,42126.0,0.0,6042.0
1,2018-10-01T00:00:00Z,51.41,41500.0,0.0,6021.0
2,2018-10-01T01:00:00Z,47.38,42353.0,0.0,6342.0
3,2018-10-01T02:00:00Z,47.59,43802.0,0.0,7144.0
4,2018-10-01T03:00:00Z,51.61,48065.0,0.0,7855.0


## 2. ENTSO-E Transparency Platform — XML

Официальная платформа европейских операторов энергосистем: цены, нагрузка, генерация. Для API нужен бесплатный токен: регистрация на transparency.entsoe.eu и запрос роли «Web API Consumer» по почте. Токен задаётся в переменной окружения `ENTSOE_TOKEN`.

В XML ответа цена лежит в элементах `Point`: номер часа (`position`) и цена (`price.amount`).

In [3]:
token = os.getenv('ENTSOE_TOKEN')
if token:
    r = requests.get('https://web-api.tp.entsoe.eu/api', params={
        'securityToken': token, 'documentType': 'A44',
        'in_Domain': '10Y1001A1001A82H', 'out_Domain': '10Y1001A1001A82H',   # код зоны DE-LU
        'periodStart': '202312312300', 'periodEnd': '202412312300'})
    entsoe = pd.read_xml(StringIO(r.text), xpath='//*[local-name()="Point"]')
    display(entsoe.head())
else:
    print('Нужен токен ENTSO-E (переменная ENTSOE_TOKEN)')

Нужен токен ENTSO-E (переменная ENTSOE_TOKEN)


## 2a. Eurostat — XML

Помесячное производство электроэнергии по видам топлива (набор `nrg_cb_pem`). В XML каждая серия (`Series`) — это страна и вид топлива, внутри неё наблюдения (`Obs`) по месяцам.

In [ ]:
url = 'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/nrg_cb_pem/M.TOTAL+CF+RA300+RA400.GWH.DE+FR+PL'
xml = requests.get(url, params={'startPeriod': '2024-01', 'endPeriod': '2024-12'}).text

fuels = {'TOTAL': 'всего', 'CF': 'ископаемое топливо', 'RA300': 'ветер', 'RA400': 'солнце'}

rows = []
for series in ET.fromstring(xml).findall('.//{*}Series'):
    key = {v.get('id'): v.get('value') for v in series.findall('{*}SeriesKey/{*}Value')}
    for obs in series.findall('{*}Obs'):
        flag = obs.find('{*}Attributes/{*}Value')
        rows.append({'country': key['geo'],
                     'fuel': fuels[key['siec']],
                     'month': obs.find('{*}ObsDimension').get('value'),
                     'gwh': float(obs.find('{*}ObsValue').get('value')),
                     'flag': flag.get('value') if flag is not None else ''})
eurostat = pd.DataFrame(rows)
eurostat.head()

,country,fuel,month,gwh,flag
0,DE,ископаемое топливо,2024-01,21883.583,e
1,DE,ископаемое топливо,2024-02,17908.115,e
2,DE,ископаемое топливо,2024-03,18652.114,e
3,DE,ископаемое топливо,2024-04,14782.617,e
4,DE,ископаемое топливо,2024-05,13684.493,e


Столбцы: страна, вид топлива (по классификатору SIEC), месяц, производство в ГВт·ч и флаг значения: `e` — оценка, `p` — предварительное, пусто — окончательное.

## 3. Energy-Charts — API, JSON

Цена на рынке на сутки вперёд и генерация по типам станций в Германии.

In [21]:
r = requests.get('https://api.energy-charts.info/price',
                 params={'bzn': 'DE-LU', 'start': '2024-01-01', 'end': '2024-12-31'}).json()
price = pd.DataFrame({'time': pd.to_datetime(r['unix_seconds'], unit='s'), 'price': r['price']})
price.head()

,time,price
0,2023-12-31 23:00:00,0.10
1,2024-01-01 00:00:00,0.01
2,2024-01-01 01:00:00,0.00
3,2024-01-01 02:00:00,-0.01
4,2024-01-01 03:00:00,-0.03


In [6]:
r = requests.get('https://api.energy-charts.info/public_power',
                 params={'country': 'de', 'start': '2024-06-01', 'end': '2024-06-07'}).json()
power = pd.DataFrame({t['name']: t['data'] for t in r['production_types']},
                     index=pd.to_datetime(r['unix_seconds'], unit='s'))
power.head()

,Hydro pumped storage consumption,Cross border electricity trading,Hydro Run-of-River,Biomass,Fossil brown coal / lignite,Fossil hard coal,Fossil oil,Fossil coal-derived gas,Fossil gas,Geothermal,...,Hydro pumped storage,Others,Waste,Wind offshore,Wind onshore,Solar,Load,Residual load,Renewable share of load,Renewable share of generation
2024-05-31 22:00:00,-32.5,9268.6,2535.8,3966.4,7029.8,353.7,377.8,650.7,2287.1,16.7,...,1429.4,112.5,888.2,3454.6,6742.4,0.0,41401.2,31204.2,42.3,60.9
2024-05-31 22:15:00,-20.3,9268.6,2518.7,3933.8,6960.7,349.3,372.8,631.3,2114.3,16.6,...,1357.1,112.5,890.0,3405.7,6840.5,0.0,41142.3,30896.1,42.4,61.3
2024-05-31 22:30:00,-24.5,9268.6,2528.6,3928.2,6939.8,345.8,380.6,614.7,2114.9,16.7,...,939.5,112.5,887.9,3421.7,6839.4,0.0,40635.7,30374.6,42.8,61.3
2024-05-31 22:45:00,-76.7,9268.6,2548.9,3935.0,6696.5,349.2,375.8,602.9,2109.9,16.7,...,708.6,112.5,887.6,3466.7,6902.3,0.0,40046.2,29677.2,43.7,62.0
2024-05-31 23:00:00,-24.8,9443.2,2508.9,3893.1,6227.5,344.6,377.2,583.4,2075.2,16.6,...,1497.9,112.5,886.9,3445.2,6801.4,0.0,39228.2,28981.5,44.5,63.2


## 4. Open-Meteo — API, JSON

Почасовая погода в трёх крупнейших городах Германии.

In [7]:
cities = ['Berlin', 'Hamburg', 'Munich']
r = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
    'latitude': '52.52,53.55,48.14', 'longitude': '13.41,9.99,11.58',
    'start_date': '2024-01-01', 'end_date': '2024-12-31',
    'hourly': 'temperature_2m,wind_speed_100m,shortwave_radiation,cloud_cover,weather_code,is_day'}).json()

weather = pd.concat([pd.DataFrame(r[i]['hourly']).assign(city=cities[i]) for i in range(3)])
weather.head()

,time,temperature_2m,wind_speed_100m,shortwave_radiation,cloud_cover,weather_code,is_day,city
0,2024-01-01T00:00,5.0,28.5,0.0,100,3,0,Berlin
1,2024-01-01T01:00,5.4,27.6,0.0,100,3,0,Berlin
2,2024-01-01T02:00,5.0,28.2,0.0,99,3,0,Berlin
3,2024-01-01T03:00,4.6,29.5,0.0,68,2,0,Berlin
4,2024-01-01T04:00,4.4,28.5,0.0,14,0,0,Berlin


## 5. Nager.Date — API, JSON

Государственные праздники Германии. Часть из них отмечается только в отдельных землях (`counties`).

In [22]:
holidays = pd.DataFrame(requests.get('https://date.nager.at/api/v3/PublicHolidays/2024/DE').json())
holidays.head()

,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2024-01-01,Neujahr,New Year's Day,DE,False,True,None,None,[Public]
1,2024-01-06,Heilige Drei Könige,Epiphany,DE,False,False,"[DE-BW, DE-BY, DE-ST]",None,[Public]
2,2024-03-08,Internationaler Frauentag,International Women's Day,DE,False,False,"[DE-BE, DE-MV]",None,[Public]
3,2024-03-29,Karfreitag,Good Friday,DE,False,True,None,None,[Public]
4,2024-03-31,Ostersonntag,Easter Sunday,DE,False,False,[DE-BB],None,[Public]


## 6. Google News — неструктурированный текст

Заголовки новостей по запросу «Strompreis» («цена на электроэнергию»). Позже из текста получаем тему и тональность.

In [23]:
r = requests.get('https://news.google.com/rss/search', params={'q': 'Strompreis', 'hl': 'de', 'gl': 'DE', 'ceid': 'DE:de'})
news = pd.read_xml(StringIO(r.text), xpath='//item')[['title', 'source', 'pubDate']]
news.head()

,title,source,pubDate
0,Von Hormus bis zu den Steckdosen in Fröndenber...,Hellweger Anzeiger,"Fri, 09 Oct 2026 04:05:00 GMT"
1,Energiemonitor: Schafft Deutschland die Energi...,DIE ZEIT,"Wed, 07 Oct 2026 08:57:30 GMT"
2,Rentner und diese Gruppe zahlen die höchsten S...,T-Online,"Mon, 05 Oct 2026 14:47:26 GMT"
3,Energie-Experiment - Strompreis fällt im Test ...,Yahoo Nachrichten,"Tue, 06 Oct 2026 14:02:54 GMT"
4,Strompreise steigen auf 3-Jahreshoch: Strompre...,agrarheute.com,"Mon, 28 Sep 2026 07:00:00 GMT"


## 7. Аналитический портрет

Из загруженных данных выбираем признаки. Первые три описывают сам объект — цену и спрос, остальные — факторы, которые на них влияют.

| Признак | Тип | Источник | Что описывает |
|:--|:--|:--|:--|
| цена электроэнергии | числовой | Energy-Charts, ENTSO-E, OPSD | **объект** |
| нагрузка (потребление) | числовой | Energy-Charts, ENTSO-E, OPSD | **объект** |
| ценовая зона | категориальный | Energy-Charts, ENTSO-E, OPSD | **объект** |
| температура | числовой | Open-Meteo | фактор: погода |
| скорость ветра | числовой | Open-Meteo | фактор: погода |
| солнечная радиация | числовой | Open-Meteo | фактор: погода |
| облачность | числовой | Open-Meteo | фактор: погода |
| тип погоды (`weather_code`) | категориальный | Open-Meteo | фактор: погода |
| город | категориальный | Open-Meteo | фактор: погода |
| выработка солнца, ветра, газа | числовой | Energy-Charts, ENTSO-E, OPSD | фактор: генерация |
| тип станции | категориальный | Energy-Charts, ENTSO-E | фактор: генерация |
| доля ВИЭ, производство по видам топлива за месяц | числовой | Eurostat | фактор: генерация |
| вид топлива, страна | категориальный | Eurostat | фактор: генерация |
| час, день недели, месяц | категориальный | время в любом источнике | фактор: поведение, сезон |
| праздник | категориальный | Nager.Date | фактор: поведение |
| издание, тема новости | категориальный | Google News | фактор: новости |
| число новостей за день | числовой | Google News | фактор: новости |

В каждом источнике есть время, не меньше 3 категориальных и 3 числовых характеристик (у праздников и новостей часть признаков вычисляется из данных).